This demonstrates an example exchange with mostly text based prompts. To be converted to code properly later.

Main goal is to test out different model / prompt combinations

In [41]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%env OPENAI_API_KEY=""
%env HUGGINGFACE_TOKEN = ""
                      
if not 'dirguard' in locals():
  %cd ..
  dirguard = True

%aimport vivabench
%aimport vivabench.util
%aimport vivabench.ontology.concepts

from vivabench.ontology import concepts
from vivabench.ontology.concepts import *
from vivabench.util import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: OPENAI_API_KEY=""
env: HUGGINGFACE_TOKEN=""


In [12]:
# set limit to number of exchanges
max_length = 512
temp = 0.3

agent_model = 'gpt-4-1106-preview'
examiner_model = 'gpt-4-1106-preview'


In [13]:
import json
with open("cases/msd_raw/msd_1.json", "r") as f:
    g = json.loads(f.read())
sample_case = ClinicalCase.from_dict(g)

In [19]:
# DDX exploration
ddx_prompt = open("vivabench/prompts/examiner_differentials.prompt", 'r').read()

agent_response = "My primary differentials are appendicitis, PID, and cancer. I am also concerned about a viral infection."

prompt = ddx_prompt.format(agent_response=agent_response, differentials=sample_case.primary_ddx.prompt())
print(prompt)

You are an examiner in a medical viva examination. 

Your student just gave you a list of medical differentials. You are to compare that to the correct answer, and output 1. true_positive: the list of differentials that are correct, 2. false_positive: differentials from the student that were not included in the answer, 3. false_negative: differentials included in the answer key that were not in the student's response.

Provide your response in json format, with {'true_positive': [List of correct differentials], 'false_positive': [List of differentials mentioned by the student but not in the answer], 'false_negative': [List of differentials from the answer not mentioned by the student]}

STUDENT RESPONSE:
My primary differentials are appendicitis, PID, and cancer. I am also concerned about a viral infection.

ANSWER:
- Appendicitis
- Pelvic inflammatory disease
- Pyelonephritis
- Sepsis
- Viral infection




In [20]:
p = str_to_msgs(prompt)

In [21]:
response = chatgpt_decode(p, model_name=examiner_model, max_length=max_length)
print(response)

```json
{
  "true_positive": ["appendicitis", "PID", "viral infection"],
  "false_positive": ["cancer"],
  "false_negative": ["pyelonephritis", "sepsis"]
}
```


In [31]:
parse_ddx_response(response)[1]['true_positive']

['appendicitis', 'PID', 'viral infection']

In [172]:
# Mistral-7B itself ain't cutting it!
model_name = "mistralai/Mistral-7B-Instruct-v0.1"
response = hfapi_decode(p, model_name=model_name, max_length=max_length)
print(response)

{
"true_positive": ["Appendicitis", "Pelvic inflammatory disease", "Viral infection"],
"false_positive": ["Cancer", "Pyelonephritis", "Sepsis"],
"false_negative": []
}


In [173]:
model_name = "mistralai/Mixtral-8x7B-Instruct-v0.1"
response = hfapi_decode(p, model_name=model_name, max_length=max_length)
print(response)

{
  "true\_positive": ["Appendicitis", "Pelvic inflammatory disease", "Viral infection"],
  "false\_positive": ["Cancer"],
  "false\_negative": ["Pyelonephritis", "Sepsis"]
}


In [15]:
# H&E investigation

bedside_prompt  = open("vivabench/prompts/examiner_bedside.prompt", 'r').read()
agent_response = "I would like to further characterise her pain, further enquire her gastrointestinal symptoms, and also take a full pregnancy history. I would also like to know if she was in Africa for the past 7 days, and if she ate pancakes for breakfast."

prompt = bedside_prompt.format(agent_response=agent_response, 
                               hopc_full=sample_case.hopc.full_history,
                               systems_review=sample_case.systems.prompt(),
                               physical_examination=sample_case.examination.prompt(),
                               pmh=sample_case.pmh.prompt())
print(prompt)

You are an examiner in a medical viva examination. 

Your examinee is describing what they would do with a patient on a bedside setting in [[EXAMINEE RESPONSE]]. Your task is to extract information to your examinee from the section [[CLINICAL INFORMATION]] below.

[[CLINICAL INFORMATION]]
HISTORY OF PRESENTING COMPLAINT:
A 26-year-old woman comes to the office because of a 3-day history of lower abdominal pain. She is 18 weeks pregnant by dates. The patient describes the pain as sharp, steady, and radiating across her lower abdomen bilaterally. Last night she developed new nausea and vomiting. She has not been able to keep down any food or drink this morning. She had a normal bowel movement yesterday. She says she felt cold and shivering this morning, followed by feeling warm; however, she did not check her temperature. She denies vaginal bleeding.

PAST MEDICAL HISTORY:
MEDICAL HISTORY: Mild intermittent asthma diagnosed in childhood requiring only occasional rescue inhaler use, no ho

In [16]:
p = str_to_msgs(prompt)
model_name = "mistralai/Mixtral-8x7B-Instruct-v0.1"
response = hfapi_decode(p, model_name=model_name, max_length=max_length)
print(response)

Based on your question, I can provide some additional information from the clinical information section.

The patient is a 26-year-old woman who is 18 weeks pregnant and presents with a 3-day history of lower abdominal pain. She describes the pain as sharp, steady, and radiating across her lower abdomen bilaterally. Last night, she developed new nausea and vomiting and has not been able to keep down any food or drink this morning. She also reports a frequent urge to urinate and a sensation of incomplete bladder emptying for the past 3 days.

On physical examination, her vital signs show a temperature of 38.8°C, pulse of 120 beats/minute, blood pressure of 110/76 mm Hg, and respiratory rate of 20/minute. She appears tired and ill but is not in distress. Her skin is hot and diaphoretic. Her abdomen is soft, diffusely tender across the lower abdomen bilaterally with no guarding, rigidity, or rebound. The uterus is gravid, non-tender, and 2 cm below the level of the umbilicus. There is pos

In [17]:
# GPT-4 still superior but Mixtral is OK?
response = chatgpt_decode(p, model_name=examiner_model, max_length=max_length)
print(response)

Based on the information provided, you can further characterize her pain as sharp, steady, and radiating across her lower abdomen bilaterally. She has been experiencing this pain for the past 3 days. Regarding her gastrointestinal symptoms, she has had a decreased appetite for 1 day and has been unable to keep any food or drink down this morning due to nausea and vomiting. She had a normal bowel movement yesterday and has not had diarrhea or constipation. Her full pregnancy history indicates that she is 18 weeks pregnant, G1P0A0, and has been attending all routine obstetric visits and testing. She has not reported any vaginal bleeding.

Regarding your specific questions, the patient has not had any recent travel, so she has not been in Africa in the past 7 days. There is no information provided about what she ate for breakfast, including whether she had pancakes.


In [33]:
# Investigations 

bedside_prompt  = open("vivabench/prompts/examiner_investigations.prompt", 'r').read()
agent_response = "I would like to further characterise her pain, further enquire her gastrointestinal symptoms, and also take a full pregnancy history. I would also like to know if she was in Africa for the past 7 days, and if she ate pancakes for breakfast."
agent_response = "I would like to do an abdominal x-ray"
agent_response = "I would like to do a FBC, electrolytes, and cardiac ultrasound, and renal ultrasound"
# agent_response = "I would like to do a FBC, EUC, and cardiac ultrasound, and renal ultrasound"

available_ix = list(sample_case.ix.keys())
prompt = bedside_prompt.format(hopc_full=sample_case.hopc.full_history,
                               agent_response=agent_response, 
                               investigations=available_ix)
print(prompt)


You are an admin parsing medical information.

A doctor is describing what he plans to do with a patient in [[DOCTOR REQUEST]]. This would include a list of investigations and other management items. You are to break down the doctor's request into a list of items. Use the same variable names for investigations present in [[AVAILABLE INVESTIGATIONS]]. If the doctor requested an investigation not included in [[AVAILABLE_INVESTIGATIONS]], include it with a similar variable name. If the doctor request does not mention any investigations, respond with an empty list []. 

[[PATIENT INFORMATION]]
A 26-year-old woman comes to the office because of a 3-day history of lower abdominal pain. She is 18 weeks pregnant by dates. The patient describes the pain as sharp, steady, and radiating across her lower abdomen bilaterally. Last night she developed new nausea and vomiting. She has not been able to keep down any food or drink this morning. She had a normal bowel movement yesterday. She says she fe

In [34]:
# Problem with Mixtral: 
# 1. It has these weird backslashes. Why? 
# 2. It cannot infer things e.g. electrolytes -> metabolic panel

p = str_to_msgs(prompt)
model_name = "mistralai/Mixtral-8x7B-Instruct-v0.1"
response = hfapi_decode(p, model_name=model_name, max_length=max_length)
print(response)

['full\_blood\_count', 'electrolytes', 'ultrasound\_renal', 'ultrasound\_cardiac']


In [35]:
response = chatgpt_decode(p, model_name=examiner_model, max_length=max_length)
print(response)

['full_blood_count', 'metabolic_panel', 'ultrasound_cardiac', 'ultrasound_renal']


In [36]:
requested_ix = [r.replace("\\", "") for r in eval(response)]

ordered_ix = set(requested_ix).intersection(set(available_ix))
overinvestigated_ix = set(requested_ix) - set(available_ix)
missed_ix = set(available_ix) - set(requested_ix)

In [40]:
print("\n".join([sample_case.ix[ix].prompt() for ix in list(ordered_ix)]))

Renal/bladder ultrasound: Renal/bladder ultrasound: No hydronephrosis noted. Incidentally noted intrauterine fetus with heartbeat of 156 beats/minute. 

FULL BLOOD COUNT
- Hemoglobin: 140 mmol/L
- WBC count: 17 x 10^9 / L
- Platelet count: 150 x 10^9 / L


METABOLIC PANEL
- Sodium: 137 mmol/L
- Potassium: 3.9 mmol/L
- Creatinine: 62 micromol/L
- Glucose: 4.3 mmol/L
- BUN: 8.2 mmol/L


